# img_det baseline - local (CPU)

Scores the published `aludam` 1.0.0 image detector on **CommunityForensics-Eval**
and writes **`notebooks/result.txt`** - that file is the deliverable.

- No training. No package changes. Nothing is committed.
- Run top to bottom: *Cell > Run All*.
- First run downloads ~750 images per class and, if missing, the 665 MB
  checkpoint (already cached on this machine).
- Scoring on CPU takes roughly 30-90 minutes for 1,500 images; it is
  **resumable** - if interrupted, just run it again.
- Every result is stamped: general-deepfake distribution, **not** validated
  for government-ID / KYC scans.

When it finishes, send me `notebooks/result.txt`.

In [ ]:
from pathlib import Path
import os
import sys


def _find_repo():
    for c in [Path.cwd(), *Path.cwd().parents]:
        if ((c / "aludam").is_dir() and (c / "models").is_dir()
                and (c / "notebooks").is_dir()):
            return c
    raise SystemExit(
        "Repo root not found. Start Jupyter inside the Deep-Fake-Analysis "
        "folder (the one containing aludam/, models/, notebooks/).")


REPO = _find_repo()
os.environ["ALUDAM_REPO"] = str(REPO)
os.environ["ALUDAM_MODELS_DIR"] = str(REPO / "models")
sys.path.insert(0, str(REPO / "notebooks"))
import img_eval_core as core
print("repo:", REPO)

In [ ]:
import importlib.util
NEED = {"numpy": "numpy", "pandas": "pandas", "PIL": "Pillow",
        "cv2": "opencv-python", "pyarrow": "pyarrow", "fsspec": "fsspec",
        "sklearn": "scikit-learn", "torch": "torch",
        "transformers": "transformers", "aludam": "aludam"}
missing = [p for m, p in NEED.items() if importlib.util.find_spec(m) is None]
if missing:
    raise SystemExit(
        "Missing packages for this kernel: %s\nInstall them first, e.g.\n"
        "  pip install %s" % (missing, " ".join(missing)))

import aludam
import sklearn
import torch
import transformers

print("aludam", aludam.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
CONFIG = dict(
    per_class=750,     # rows per class (plan default 750)
    seed=20261004,     # selection + split seed
    device="auto",     # "auto" = CUDA when available, else CPU
    workers=8,         # parallel metadata readers (HF-friendly)
    score_every=25,    # progress log interval, in images
)
print(CONFIG)

In [ ]:
# Step 1: metadata-only index of the dataset (payload column never read).
index_rows = core.build_index(REPO, workers=CONFIG["workers"])
print("indexed rows:", len(index_rows))

In [ ]:
# Step 2: label evidence + source-grouped dev/cal/test split.
selected, splits, evidence, warns = core.select_and_split(
    index_rows, per_class=CONFIG["per_class"], seed=CONFIG["seed"])
print("selected:", len(selected), "| warnings:", len(warns))

In [ ]:
# Step 3: download ONLY the selected images (resumable; re-run skips done).
PAYLOAD_DIR = REPO / "aludam" / "eval" / "data" / "index" / "img_payload"
payload_stats = core.fetch_payload(selected, PAYLOAD_DIR)
payload_stats

In [ ]:
# Step 4: load aludam img_det (weights are fetched once if missing).
detector, device_used, device_notes = core.get_detector(CONFIG["device"])
print("device:", device_used)
for n in device_notes:
    print(" -", n)

In [ ]:
# Step 5: score every selected image (resumable JSONL; re-run skips done).
SCORES_PATH = REPO / "aludam" / "eval" / "data" / "index" / "img_scores.jsonl"
score_stats = core.score_all(detector, selected, PAYLOAD_DIR, SCORES_PATH,
                             every=CONFIG["score_every"])
score_stats

In [ ]:
# Step 6: metrics -> per-signal diagnosis -> dual-budget fit -> test verify.
OUT_PATH = REPO / "notebooks" / "result.txt"
text = core.analyze_and_write(
    repo=REPO, selected=selected, splits=splits, evidence=evidence,
    warnings_=warns, payload_stats=payload_stats, score_stats=score_stats,
    scores_path=SCORES_PATH, out_path=OUT_PATH,
    per_class=CONFIG["per_class"], seed=CONFIG["seed"],
    device_used=device_used, device_notes=device_notes)
print(text)
print("RESULT FILE:", OUT_PATH)

## Done

`notebooks/result.txt` now holds:

1. environment + dataset revision + label-convention evidence,
2. dev-split per-signal AUROC (score vs model_signal_raw vs each family),
3. the diagnosis bucket (calibration / fusion dilution / model ceiling),
4. the dual-budget band fitted on **calibration only** (FNR <= 1%, FPR <= 5%),
5. one-shot test verification with Wilson upper bounds and PASS/FAIL.

Send that file back. Re-running any cell is safe - downloads and scores
resume where they stopped.